In [1]:
import sys
from pathlib import Path

# اضافه کردن src به path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

print("Project root:", project_root)
print("Python path:", sys.path[0])

# تست import ماژول‌ها
try:
    from legal_multi_agent.state.schemas import MASharedState
    from legal_multi_agent.state.constants import CATEGORY_TO_DOMAIN
    from legal_multi_agent.graphs.main_graph import graph
    print("✅ All imports successful!")
except Exception as e:
    print(f"❌ Import error: {e}")
    raise

Project root: f:\Thesis\project\3-Multi-Agent-System
Python path: f:\Thesis\project\3-Multi-Agent-System\src
✅ All imports successful!


In [2]:
# چک کردن نودهای گراف
print("Graph nodes:")
for node_name in graph.nodes:
    print(f"  - {node_name}")

print("\nGraph compiled successfully!" if graph else "❌ Graph is None")


Graph nodes:
  - __start__
  - supervisor
  - researcher
  - reasoner
  - critic
  - finalize

Graph compiled successfully!


In [3]:
def print_state_summary(state: dict, title: str = "State Summary"):
    """نمایش خلاصه state به شکل خوانا"""
    print(f"\n{'='*60}")
    print(f"  {title}")
    print('='*60)
    
    # اطلاعات سوال
    print(f"\n📋 Question Info:")
    print(f"  Number: {state.get('question_number')}")
    print(f"  Category: {state.get('category')}")
    print(f"  Domain: {state.get('domain')}")
    
    # وضعیت RAG
    if state.get('context'):
        print(f"\n🔍 RAG Status:")
        print(f"  Context length: {len(state.get('context', ''))} chars")
        print(f"  Retrieved docs: {len(state.get('rag_results', []))} items")
        print(f"  Docs with article: {len([d for d in state.get('docs_meta', []) if d.get('article_number')])}")
    
    # وضعیت Reasoner
    if state.get('draft_toon'):
        print(f"\n💭 Reasoner Output:")
        dt = state['draft_toon']
        print(f"  Answer: {dt.get('answer')}")
        print(f"  Confidence: {dt.get('confidence')}")
        print(f"  Explanation: {dt.get('explanation', '')[:80]}...")
    
    # وضعیت Critic
    if state.get('critic_toon'):
        print(f"\n⚖️  Critic Feedback:")
        ct = state['critic_toon']
        print(f"  Needs revision: {ct.get('needs_revision')}")
        print(f"  Issue: {ct.get('issue')}")
        print(f"  Action: {ct.get('action')}")
    
    # خروجی نهایی
    if state.get('final_toon'):
        print(f"\n✅ Final Answer:")
        ft = state['final_toon']
        print(f"  Answer: {ft.get('answer')}")
        print(f"  Confidence: {ft.get('confidence')}")
        print(f"  Revision count: {state.get('revision_count', 0)}")
    
    print(f"\n{'='*60}\n")


In [ ]:
import pandas as pd
import json
import re

# ============ لود سوال ============
QUESTIONS_PATH = r"F:\Thesis\project\403-vekalat\structured_questions.csv"
df_q = pd.read_csv(QUESTIONS_PATH)

TEST_QUESTION_NUMBER = 5  

row = df_q[df_q["question_number"] == TEST_QUESTION_NUMBER].iloc[0]

print(f"Testing with Question #{TEST_QUESTION_NUMBER}")
print(f"Category: {row['category']}")
print(f"\nQuestion: {row['question']}")
print(f"\nRaw Options: {row['options']}\n")

# ============ توابع کمکی ============
def to_list(opts):
    if isinstance(opts, list): 
        return opts
    if isinstance(opts, str):
        try:
            v = json.loads(opts)
            if isinstance(v, list): 
                return v
        except Exception:
            pass
        for sep in ["|", "\n", "؛", ";", "/", "\\", "،"]:
            if sep in opts:
                return [x.strip() for x in opts.split(sep) if x.strip()]
        return [opts.strip()]
    return [str(opts)]

def render_numeric_options(opts):
    clean = []
    for o in opts:
        m = re.match(r"^\s*\d+\)\s*(.+)$", o)
        if m:
            o = m.group(1).strip()
        clean.append(o)
    return "\n".join(f"{i+1}) {o}" for i, o in enumerate(clean))

# ============ آماده‌سازی ورودی ============
question_number = int(row["question_number"])
category = str(row["category"]).strip()
question = str(row["question"]).strip()
opts_raw = row["options"]

options_list = to_list(opts_raw)
options_text = render_numeric_options(options_list)
domain = CATEGORY_TO_DOMAIN.get(category, "unknown")

print(f"Prepared options:\n{options_text}")
print(f"\nDomain: {domain}\n")

# ============ ساخت state ورودی ============
state_input = {
    "question_number": question_number,
    "category": category,
    "domain": domain,
    "question": question,
    "options_text": options_text,
    "max_revisions": 2,
    "revision_count": 0,
}

print("🚀 Starting graph execution...")
print("="*60)

# ============ اجرای گراف ============
try:
    output = graph.invoke(state_input)
    print("✅ Graph execution completed successfully!")
except Exception as e:
    print(f"❌ Graph execution failed: {e}")
    import traceback
    traceback.print_exc()
    raise

# ============ نمایش خلاصه ============
print_state_summary(output, f"Results for Question #{question_number}")


Testing with Question #5
Category: حقوق مدنی

Question: شرط وکالت زن در طلاق در چه قالب هایی قابل تحقق است؟

Raw Options: 1) به شکل شرط نتیجه یا شرط فعل، ضمن عقد نکاح | 2) به شکل شرط فعل، ضمن عقد نکاح یا عقد دیگری | 3) به شکل شرط نتیجه یا فعل ضمن عقد نکاح یا عقد دیگری | 4) به شکل شرط نتیجه ضمن عقد نکاح یا عقد دیگری

Prepared options:
1) به شکل شرط نتیجه یا شرط فعل، ضمن عقد نکاح
2) به شکل شرط فعل، ضمن عقد نکاح یا عقد دیگری
3) به شکل شرط نتیجه یا فعل ضمن عقد نکاح یا عقد دیگری
4) به شکل شرط نتیجه ضمن عقد نکاح یا عقد دیگری

Domain: civil

🚀 Starting graph execution...
📝 Query: شرط وکالت زن در طلاق در چه قالب هایی قابل تحقق است؟...
🔍 روش انتخاب شده: Semantic search
🔍 Semantic only: 20 نتیجه
   ✓ Retrieved: 20 documents
🔄 Reranking 20 سند (بدون law+article)...
   ✓ Reranked: top 5 documents
✅ Graph execution completed successfully!

  Results for Question #5

📋 Question Info:
  Number: 5
  Category: حقوق مدنی
  Domain: civil

🔍 RAG Status:
  Context length: 7922 chars
  Retrieved docs: 5 ite

In [5]:
print("🔍 Detailed Debug Information")
print("="*60)

# Context preview
if output.get('context_preview'):
    print("\n📄 Context Preview (first 500 chars):")
    print(output['context_preview'][:500])
    print("...")

# Docs metadata
if output.get('docs_meta'):
    print(f"\n📚 Retrieved Documents ({len(output['docs_meta'])} items):")
    for i, doc in enumerate(output['docs_meta'][:5], 1):
        print(f"  {i}. Article: {doc.get('article_number')}, "
              f"Law: {doc.get('law')}, "
              f"Type: {doc.get('source_type')}")

# Draft raw
if output.get('draft_raw'):
    print("\n📝 Draft Raw Output:")
    print(output['draft_raw'])

# Critic raw
if output.get('critic_raw'):
    print("\n⚖️  Critic Raw Output:")
    print(output['critic_raw'])

# Final raw
if output.get('final_raw'):
    print("\n✅ Final Raw Output:")
    print(output['final_raw'])


🔍 Detailed Debug Information

📄 Context Preview (first 500 chars):
[منبع 1] (وحدت رویه) - قانون مدنی
title: عدم امکان اعمال وکالت زوجه در طلاق در صورت اثبات عدم تمکین بدون مانع مشروع
issuer: هیات‌ عمومی دیوان ‌عالی ‌کشور
vote_text: نظر به اینکه مطابق ماده 1108 قانون مدنی تمکین از زوج تکلیف قانونی زوجه است، بنابراین در صورتی که بدون مانع مشروع از ادای وظایف زوجیت امتناع و زوج این امر را در دادگاه اثبات و با اخذ اجازه از دادگاه همسر دیگری اختیار نماید، وکالت زوجه از زوج در طلاق که به حکم ماده 1119 قانون مدنی ضمن عقد نکاح شرط و مراتب در سند ازدواج ذیل بند ب شرایط 
...

📚 Retrieved Documents (5 items):
  1. Article: None, Law: None, Type: None
  2. Article: None, Law: None, Type: None
  3. Article: 1119, Law: None, Type: None
  4. Article: None, Law: None, Type: None
  5. Article: None, Law: None, Type: None

📝 Draft Raw Output:
results{explanation,answer,confidence}:
بر اساس ماده 1119 قانون مدنی، شرط وکالت زن در طلاق می‌تواند به صورت شرط نتیجه یا شرط فعل در ضمن عقد نکاح یا عقد لازم دیگری 

In [6]:
# اگر می‌خواهی با پاسخ صحیح مقایسه کنی
GOLD_PATH = r"F:\Thesis\project\1-BaselineTest\GOLD\Gold.csv"

try:
    df_gold = pd.read_csv(GOLD_PATH)
    
    # حذف سوال 89
    if 89 in df_gold["idx"].values:
        df_gold = df_gold[df_gold["idx"] != 89].reset_index(drop=True)
    
    gold_row = df_gold[df_gold["idx"] == question_number]
    
    if not gold_row.empty:
        gold_answer = str(gold_row["Gold"].iloc[0]).strip()
        predicted = output.get("final_toon", {}).get("answer")
        
        print("\n🎯 Answer Comparison:")
        print(f"  Gold answer: {gold_answer}")
        print(f"  Predicted: {predicted}")
        print(f"  Match: {'✅ CORRECT' if str(predicted) == gold_answer else '❌ WRONG'}")
    else:
        print(f"\n⚠️  No gold answer found for question #{question_number}")
        
except Exception as e:
    print(f"\n⚠️  Could not load gold answers: {e}")



🎯 Answer Comparison:
  Gold answer: 3
  Predicted: 3
  Match: ✅ CORRECT
